# Avito: приватный GPU-запуск
Подготовлен, но ещё не выполнен в Colab/Kaggle. Выберите GPU и включите интернет. Не публикуйте исходные объявления, результаты или токены. Используйте source-bundle из scripts/package_notebook.py и отдельный исходный ZIP. Сервер доступен только внутри notebook; публичных туннелей нет.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, zipfile, hashlib, time, urllib.request
BASE = Path('/kaggle/working' if Path('/kaggle/working').exists() else '/content')
# Измените два пути на свои приватные файлы. В Colab можно загрузить через панель Files.
SOURCE_ZIP = Path('/content/avito-source.zip')
DATA_ZIP = Path('/content/innoglobalhack_avito(1).zip')
WORK = BASE / 'avito-hackathon'
assert SOURCE_ZIP.is_file(), 'Укажите source-bundle ZIP'
assert DATA_ZIP.is_file(), 'Укажите приватный ZIP данных'
WORK.mkdir(exist_ok=True)
with zipfile.ZipFile(SOURCE_ZIP) as z:
    for item in z.infolist():
        target = (WORK / item.filename).resolve()
        assert target.is_relative_to(WORK.resolve()), 'Unsafe ZIP path'
        if item.is_dir(): continue
        target.parent.mkdir(parents=True, exist_ok=True)
        content=z.read(item)
        if target.exists(): assert target.read_bytes()==content, 'Existing source differs; preserve it'
        else:
            with target.open('xb') as f: f.write(content)
os.chdir(WORK)
subprocess.run(['nvidia-smi'], check=True)


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','--target','.tools/python','-r','requirements-prototype.txt'],check=True)
sys.path.insert(0, str(WORK))
sys.path.insert(0, str(WORK/'.tools/python'))
from scripts.setup_local_llm import fetch, TAG_URL, TAG_SHA256, MODEL_NAME, MODEL_SHA256, MODEL_REVISION
# Выбранный официальный nightly закреплён SHA256 указателя, модель — ревизией и LFS SHA256.
tag_file=fetch(TAG_URL, WORK/'.tools/downloads/nightly-tag.txt', TAG_SHA256)
tag=tag_file.read_text().strip()
assert __import__('re').fullmatch(r'b\d+',tag), 'Unexpected tag'
source=WORK/'.tools/llama.cpp'
if not source.exists():
    subprocess.run(['git','clone','--depth','1','--branch',tag,'https://github.com/ggml-org/llama.cpp',str(source)],check=True)
else:
    actual=subprocess.check_output(['git','-C',str(source),'describe','--tags','--exact-match'],text=True).strip()
    assert actual==tag, 'Existing runtime source differs'
build=WORK/'.tools/llama-build'
subprocess.run(['cmake','-S',str(source),'-B',str(build),'-DGGML_CUDA=ON','-DLLAMA_CURL=OFF','-DLLAMA_BUILD_TESTS=OFF'],check=True)
subprocess.run(['cmake','--build',str(build),'--config','Release','--target','llama-server','-j','2'],check=True)
model=fetch(f'https://huggingface.co/Qwen/Qwen3-8B-GGUF/resolve/{MODEL_REVISION}/{MODEL_NAME}',WORK/'data/models'/MODEL_NAME,MODEL_SHA256)
print({'runtime_tag':tag,'model_revision':MODEL_REVISION,'model_sha256':MODEL_SHA256})


In [ ]:
# Повторно не запускает сервер автоматически. Живой процесс сохраняется в server.
assert 'server' not in globals() or server.poll() is not None, 'Сервер уже работает'
log_path=WORK/'data/llama-server.log'
log_path.parent.mkdir(parents=True,exist_ok=True)
log=log_path.open('a',encoding='utf-8')
server=subprocess.Popen([str(build/'bin/llama-server'),'-m',str(model),'--alias','avito-qwen3-8b','--host','127.0.0.1','--port','8080','--ctx-size','8192','--parallel','1','--n-gpu-layers','99','--jinja','--chat-template-kwargs',json.dumps({'enable_thinking':False})],stdout=log,stderr=subprocess.STDOUT)
for attempt in range(120):
    if server.poll() is not None: raise RuntimeError('Runtime exited; inspect data/llama-server.log')
    try:
        with urllib.request.urlopen('http://127.0.0.1:8080/health',timeout=2) as response:
            if response.status==200: break
    except Exception: time.sleep(2)
else: raise TimeoutError('Model not ready; inspect log, do not start a duplicate process')
print('GPU server is ready')


In [ ]:
from avito.llm import analyze
started=time.perf_counter()
result=analyze('Выполняем штукатурку стен. Электрику не делаем.',101)
print(json.dumps(result,ensure_ascii=False,indent=2))
print('full_request_seconds',time.perf_counter()-started)
# Это один smoke-тест, а не оценка качества или устойчивого p95.


In [ ]:
# Только development, никогда не подбирайте по control.
subprocess.run([sys.executable,'scripts/prepare_review.py','--archive',str(DATA_ZIP)],check=True)
subprocess.run([sys.executable,'scripts/train_baseline.py','--archive',str(DATA_ZIP),'--output-dir','data/runs/baseline-cloud'],check=True)
# Ручная разметка ещё требуется; baseline метрики по автометкам являются noisy-label диагностикой.


## Завершение
Скачайте нужные приватные результаты и сохраните их команде. Затем остановите процесс следующей ячейкой и завершите GPU-сессию в интерфейсе Colab/Kaggle. Не публикуйте содержимое data/ и контрольные примеры. Это остановка нашего сервера, без удаления файлов.


In [ ]:
if 'server' in globals() and server.poll() is None:
    server.terminate()
    server.wait(timeout=20)
if 'log' in globals(): log.close()
